# Travaux Pratiques : Régression Linéaire & Régression Logistique
**Filières : IISD & IDRS — Semestre 2**

---
## Partie 1 — Régression Linéaire
### Objectif
Construire un modèle de régression linéaire pour prédire la consommation de carburant (MPG) d'automobiles des années 70-80 à partir de leurs caractéristiques techniques.

### 1.1 — Importation des bibliothèques

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

%matplotlib inline
print("Bibliothèques importées avec succès.")

### 1.2 — Chargement du jeu de données Auto MPG

On utilise le jeu de données **Auto MPG** classique. Il décrit des automobiles de la fin des années 70 et du début des années 80, avec des attributs tels que : cylindres, déplacement, puissance et poids.

In [ ]:
column_names = ['MPG','Cylinders','Displacement','Horsepower','Weight',
                'Acceleration', 'Model Year', 'Origin']

raw_dataset = pd.read_csv(
    'auto-mpg.csv',
    usecols=range(8),
    names=column_names,
    na_values='?',
    header=0
)

dataSet = raw_dataset.copy()
print("Shape :", dataSet.shape)
dataSet.head()

### 1.3 — Prétraitement des données

Vérification des valeurs manquantes avec `isnull()` :

In [ ]:
print("Valeurs manquantes par colonne :")
print(dataSet.isnull().sum())

In [ ]:
# Suppression des lignes avec valeurs manquantes
dataSet = dataSet.dropna()
print("Après suppression des NaN :")
print(dataSet.isnull().sum())
print("Nouvelle shape :", dataSet.shape)

### 1.4 — Analyse exploratoire

In [ ]:
# Distribution de MPG
plt.figure(figsize=(7, 4))
sns.histplot(dataSet['MPG'], kde=True, color='steelblue')
plt.title('Distribution de MPG')
plt.xlabel('MPG')
plt.ylabel('Fréquence')
plt.tight_layout()
plt.show()
print("Les valeurs de MPG sont distribuées de façon quasi-normale avec quelques valeurs aberrantes.")

In [ ]:
# Matrice de corrélation
cor = dataSet.corr()
plt.figure(figsize=(9, 7))
sns.heatmap(cor, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Matrice de corrélation — Auto MPG')
plt.tight_layout()
plt.show()
print("\nMatrice de corrélation :")
cor

**Observations :**
- `Weight` a la corrélation négative la plus forte avec `MPG` (-0.83) : les voitures plus lourdes consomment plus.
- `Displacement`, `Cylinders` et `Horsepower` sont également fortement corrélés négativement avec MPG.
- `Model Year` et `Origin` ont une corrélation positive avec MPG.

### 1.5 — Fractionnement Train / Test

In [ ]:
# Variable cible et features
X = dataSet.drop('MPG', axis=1)
Y = dataSet['MPG']

# 70% entraînement, 30% test
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, random_state=42
)

print(f"Taille entraînement : {X_train.shape}")
print(f"Taille test         : {X_test.shape}")

### 1.6 — Génération du modèle de régression linéaire

In [ ]:
lin_model = LinearRegression()
lin_model.fit(X_train, Y_train)

print("Coefficients du modèle :")
for feat, coef in zip(X.columns, lin_model.coef_):
    print(f"  {feat:15s} : {coef:.4f}")
print(f"\nIntercepte : {lin_model.intercept_:.4f}")

### 1.7 — Évaluation du modèle

In [ ]:
predictions = lin_model.predict(X_test)

# Visualisation prédictions vs réalité
plt.figure(figsize=(7, 5))
plt.scatter(Y_test, predictions, alpha=0.6, color='steelblue', edgecolors='k', linewidths=0.3)
plt.plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()], 'r--', lw=2, label='Idéal')
plt.xlabel('MPG réel')
plt.ylabel('MPG prédit')
plt.title('Régression Linéaire — Prédictions vs Valeurs réelles')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
mse       = mean_squared_error(Y_test, predictions)
rmse      = np.sqrt(mse)
r2        = r2_score(Y_test, predictions)

print(f'MSE  (Erreur quadratique moyenne) : {mse:.4f}')
print(f'RMSE (Racine MSE)                 : {rmse:.4f}')
print(f'R²   (Coefficient de détermination): {r2:.4f}')

**Interprétation :**
- Un R² ≈ 0.84 signifie que le modèle explique **84%** de la variance dans la consommation de carburant.
- Le RMSE indique l'erreur de prédiction moyenne en MPG.

---
## Partie 2 — Régression Logistique
### Objectif
Prédire si un étudiant sera **admis** dans un programme universitaire en fonction de son score GRE, sa moyenne GPA et le prestige de son école.

### 2.1 — Importation des bibliothèques

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from sklearn.metrics import confusion_matrix, classification_report

print("Bibliothèques chargées.")

### 2.2 — Chargement du jeu de données binary.csv

Données de régression logistique fournies par l'**UCLA** (Institut de recherche et d'éducation numériques).

| Variable | Description |
|----------|-------------|
| `admit`  | 1 = admis, 0 = non admis (variable cible) |
| `gre`    | Score au Graduate Record Examination |
| `gpa`    | Grade Point Average (moyenne académique) |
| `prestige` | Niveau de prestige de l'établissement (1=plus prestigieux, 4=moins) |

In [ ]:
dataset = pd.read_csv('binary.csv')
dataset.columns = ['admit', 'gre', 'gpa', 'prestige']

print("Shape :", dataset.shape)
dataset.head()

### 2.3 — Analyse exploratoire

In [ ]:
# Distribution des admissions
print("Distribution de la variable cible 'admit' :")
print(dataset['admit'].value_counts())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sns.histplot(data=dataset, x='gre', hue='admit', kde=True, ax=axes[0], palette=['salmon','steelblue'])
axes[0].set_title('GRE par admission')

sns.histplot(data=dataset, x='gpa', hue='admit', kde=True, ax=axes[1], palette=['salmon','steelblue'])
axes[1].set_title('GPA par admission')

sns.countplot(data=dataset, x='prestige', hue='admit', ax=axes[2], palette=['salmon','steelblue'])
axes[2].set_title('Prestige par admission')

plt.tight_layout()
plt.show()

### 2.4 — Prétraitement : valeurs manquantes et encodage

In [ ]:
print("Valeurs manquantes :")
print(dataset.isnull().sum())

In [ ]:
# Encodage one-hot de la variable catégorielle 'prestige'
dummy_ranks = pd.get_dummies(dataset['prestige'], prefix='prestige')
print("Variables dummy (5 premières lignes) :")
dummy_ranks.head()

In [ ]:
# Construction du dataset final pour la régression
cols = ['admit', 'gre', 'gpa']
data = dataset[cols].join(dummy_ranks)

print("Dataset final (shape) :", data.shape)
data.head()

### 2.5 — Fractionnement Train / Test (70% / 30%)

In [ ]:
x = data.drop('admit', axis=1)
y = data['admit']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=1
)

print(f"Entraînement : {x_train.shape} | Test : {x_test.shape}")

### 2.6 — Génération du modèle de régression logistique

In [ ]:
logmodel = LogisticRegression(C=1e20, max_iter=1000)
logmodel.fit(x_train, y_train)

print("Intercepte :", logmodel.intercept_)
print("\nCoefficients :")
for feat, coef in zip(x.columns, logmodel.coef_[0]):
    print(f"  {feat:15s} : {coef:.6f}")

**Fonction score (logit) :**

$$z = \beta_0 + \beta_1 \cdot gre + \beta_2 \cdot gpa + \beta_3 \cdot p_1 + \beta_4 \cdot p_2 + \beta_5 \cdot p_3 + \beta_6 \cdot p_4$$

$$P(\text{admit}=1) = \frac{1}{1 + e^{-z}}$$

### 2.7 — Évaluation du modèle

In [ ]:
y_pred = logmodel.predict(x_test)

count_misclassified = (y_test != y_pred).sum()
accuracy = metrics.accuracy_score(y_test, y_pred)

print(f"Échantillons mal classés : {count_misclassified}")
print(f"Précision (Accuracy)     : {accuracy:.4f}  ({accuracy*100:.1f}%)")

In [ ]:
# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non admis (0)', 'Admis (1)'],
            yticklabels=['Non admis (0)', 'Admis (1)'])
plt.title('Matrice de confusion — Régression Logistique')
plt.ylabel('Valeur réelle')
plt.xlabel('Valeur prédite')
plt.tight_layout()
plt.show()

print("\nRapport de classification :")
print(classification_report(y_test, y_pred, target_names=['Non admis', 'Admis']))

### 2.8 — Visualisation de la probabilité prédite

In [ ]:
y_proba = logmodel.predict_proba(x_test)[:, 1]

plt.figure(figsize=(8, 4))
plt.hist(y_proba[y_test == 0], bins=20, alpha=0.6, color='salmon',  label='Non admis (réel)')
plt.hist(y_proba[y_test == 1], bins=20, alpha=0.6, color='steelblue', label='Admis (réel)')
plt.axvline(0.5, color='black', linestyle='--', label='Seuil = 0.5')
plt.xlabel('Probabilité prédite d\'admission')
plt.ylabel('Nombre d\'étudiants')
plt.title('Distribution des probabilités prédites par classe réelle')
plt.legend()
plt.tight_layout()
plt.show()

---
## Conclusion

| Modèle | Jeu de données | Métrique | Résultat |
|--------|---------------|----------|----------|
| Régression Linéaire | Auto MPG | R² | ~0.84 |
| Régression Logistique | Binary (UCLA) | Accuracy | ~0.74 |

**Régression linéaire :** Le modèle explique ~84% de la variance de MPG. Le poids (`Weight`) est le prédicteur le plus important. Les points s'alignent bien autour de la droite idéale dans le graphe prédiction/réalité.

**Régression logistique :** Avec ~74% de précision, le modèle prédit correctement l'admission dans 3 cas sur 4. Les variables `gpa` et le prestige de l'établissement sont les facteurs les plus déterminants. Le modèle souffre d'un déséquilibre de classes (273 refusés vs 127 admis), ce qui explique en partie les faux négatifs.